In [1]:
import ibis
from ibis import _, selectors as s
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

t_distances = con.table("working_distance_ttwa_km")
t_fixed = con.table("working_fixed")

In [6]:
# Pick 2 random firms from t_distances
random_ids = (
    t_distances
    .limit(1000)
    .order_by(ibis.random())
    .distinct(on='firm_i')
    .limit(2)
    .execute()
)
ids = random_ids['firm_i'].tolist()
print(ids)

chain_lengths = [7, 13]
chains = []
for index, firm_i in enumerate(ids):
    ch = chain_lengths[index]

    # Find an id that is 7 spatial steps away from firm_i
    curr_chain = [firm_i]
    curr_id = firm_i
    while len(curr_chain) <= ch:
        print(curr_chain)
        firm_j = (
            t_distances
            .filter(_.firm_i == curr_id)
            .order_by(ibis.random())
            .limit(1)
        )
        curr_id = firm_j.execute()['firm_j'].iloc[0]
        curr_chain.append(curr_id)
    t_distance_filtered = (
        t_distances
        .filter(
            (_.firm_i.isin(curr_chain)) &
            (_.firm_j.isin(curr_chain))
        )
    )
    chains.append({
        'ids': curr_chain,
        'd_table': t_distance_filtered.execute()
    })
print(chains)

['06403571', '00416937']
['06403571']
['06403571', '03708355']
['06403571', '03708355', '05857493']
['06403571', '03708355', '05857493', '10181910']
['06403571', '03708355', '05857493', '10181910', '05209183']
['06403571', '03708355', '05857493', '10181910', '05209183', '03645630']
['06403571', '03708355', '05857493', '10181910', '05209183', '03645630', '03777879']


IOException: IO Error: Could not read from file "/mnt/c/Users/lazym/Documents/Code/dissertation/build/output/fame_data.duckdb": Cannot allocate memory